# 🎬 דיבוב לעברית עם הקולות המקוריים + העברת רגש ואינטונציה (`Seed-VC` V3) – אווטאר: שבעת המקלטים

### 🌟 איך עובדת גרסת V3 (המרת קול ורגש פר-משפט)?
1. **עברית ברורה ב-100% (אפס מילים מחורבשות):** המילים בעברית נלקחות מתוך 290 קובצי העברית הנקיים שכבר עברו בקרת איכות ותיקון הגייה מלא (`dubbed_lines_clean_he/`).
2. **הלבשת הקול המקורי + הרגש המדויק של כל סצנה (`Seed-VC 44.1kHz F0 + RMVPE`):**
   * לכל משפט מתוך ה-290, המודל טוען את **המשפט המקורי באנגלית מאותה שנייה בפרק (`orig_lines_en/`)** יחד עם דגימת הקול של הדמות (`voice_refs/`).
   * מודל `RMVPE` מחלץ את **עקומת האינטונציה, גובה הקול והדינמיקה הרגשית** של השחקן המקורי באותו משפט בדיוק (צעקה, לחישה, פחד, התרגשות) ומעביר אותה למשפט העברי.
   * מודל ה-Diffusion **`Seed-VC` (`BigVGAN 44.1kHz`)** מייצר מחדש את המשפט בעברית מתוך מיתרי הקול והרגש של השחקן המקורי!

### 🚀 הוראות הרצה:
בתפריט העליון לחצו על **Runtime -> Restart session and run all**!

In [ ]:
# @title 1. התקנת Seed-VC וטעינת מודל ה-Diffusion + RMVPE F0 + BigVGAN 44.1kHz ל-GPU
import os
if not os.path.exists('/content/seed-vc'):
    !git clone -q https://github.com/Plachtaa/seed-vc.git /content/seed-vc

%cd /content/seed-vc
!pip install -q munch einops librosa soundfile torchaudio transformers huggingface_hub pyyaml descript-audio-codec pydub hydra-core

import sys
sys.path.insert(0, '/content/seed-vc')

import warnings
warnings.simplefilter('ignore')
import json
import yaml
import importlib
import torch
import torch.serialization
importlib.reload(torch.serialization)
_real_torch_load = torch.serialization.load
def _safe_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _real_torch_load(*args, **kwargs)
torch.load = _safe_torch_load
import torchaudio
import librosa
import soundfile as sf
import numpy as np
from modules.commons import build_model, load_checkpoint, recursive_munch
from hf_utils import load_custom_model_from_hf
from modules.rmvpe import RMVPE
from modules.campplus.DTDNN import CAMPPlus
from modules.bigvgan import bigvgan
from modules.audio import mel_spectrogram
from transformers import AutoFeatureExtractor, WhisperModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Running on device: {device}')

# טעינת מודל Seed-VC 44.1kHz עם תמיכה ב-F0 (אינטונציה ורגש)
dit_ckpt, dit_cfg = load_custom_model_from_hf(
    'Plachta/Seed-VC',
    'DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema_v2.pth',
    'config_dit_mel_seed_uvit_whisper_base_f0_44k.yml'
)
config = yaml.safe_load(open(dit_cfg, 'r'))
model_params = recursive_munch(config['model_params'])
model_params.dit_type = 'DiT'
vc_model = build_model(model_params, stage='DiT')
sr = config['preprocess_params']['sr']  # 44100
hop_length = config['preprocess_params']['spect_params']['hop_length']  # 512

vc_model, _, _, _ = load_checkpoint(vc_model, None, dit_ckpt, load_only_params=True, ignore_modules=[], is_distributed=False)
for k in vc_model:
    vc_model[k].eval().to(device)
vc_model.cfm.estimator.setup_caches(max_batch_size=1, max_seq_length=8192)

# טעינת RMVPE לחילוץ והעברת אינטונציה ורגש (F0)
rmvpe_path = load_custom_model_from_hf('lj1995/VoiceConversionWebUI', 'rmvpe.pt', None)
f0_extractor = RMVPE(rmvpe_path, is_half=False, device=device)
f0_fn = f0_extractor.infer_from_audio

# טעינת CAMPPlus לחילוץ חתימת הקול של הדמות
camp_ckpt = load_custom_model_from_hf('funasr/campplus', 'campplus_cn_common.bin', config_filename=None)
campplus_model = CAMPPlus(feat_dim=80, embedding_size=192)
campplus_model.load_state_dict(torch.load(camp_ckpt, map_location='cpu'))
campplus_model.eval().to(device)

# טעינת BigVGAN 44.1kHz Vocoder
from huggingface_hub import hf_hub_download
bvg_cfg = hf_hub_download(repo_id=model_params.vocoder.name, filename='config.json')
bvg_pts = hf_hub_download(repo_id=model_params.vocoder.name, filename='bigvgan_generator.pt')
h_bvg = bigvgan.load_hparams_from_json(bvg_cfg)
bigvgan_model = bigvgan.BigVGAN(h_bvg, use_cuda_kernel=False)
bigvgan_model.load_state_dict(torch.load(bvg_pts, map_location='cpu')['generator'])
bigvgan_model.remove_weight_norm()
vocoder_fn = bigvgan_model.eval().to(device)

# טעינת Whisper Content Encoder
whisper_name = model_params.speech_tokenizer.name
whisper_dtype = torch.float16 if device.type == 'cuda' else torch.float32
whisper_model = WhisperModel.from_pretrained(whisper_name, torch_dtype=whisper_dtype).to(device)
del whisper_model.decoder
whisper_feature_extractor = AutoFeatureExtractor.from_pretrained(whisper_name)

@torch.no_grad()
def semantic_fn(waves_16k):
    ori_inputs = whisper_feature_extractor([waves_16k.squeeze(0).cpu().numpy()], return_tensors='pt', return_attention_mask=True, sampling_rate=16000)
    ori_input_features = whisper_model._mask_input_features(ori_inputs.input_features, attention_mask=ori_inputs.attention_mask).to(device)
    ori_outputs = whisper_model.encoder(ori_input_features.to(whisper_model.encoder.dtype), return_dict=True)
    S_ori = ori_outputs.last_hidden_state.to(torch.float32)
    return S_ori[:, :waves_16k.size(-1) // 320 + 1]

mel_fn_args = {
    'n_fft': config['preprocess_params']['spect_params']['n_fft'],
    'win_size': config['preprocess_params']['spect_params']['win_length'],
    'hop_size': hop_length,
    'num_mels': config['preprocess_params']['spect_params']['n_mels'],
    'sampling_rate': sr,
    'fmin': config['preprocess_params']['spect_params'].get('fmin', 0),
    'fmax': None if config['preprocess_params']['spect_params'].get('fmax', 'None') == 'None' else 8000,
    'center': False
}
mel_fn = lambda x: mel_spectrogram(x, **mel_fn_args)
print('✅ Seed-VC 44.1kHz + RMVPE Emotion F0 + BigVGAN loaded successfully!')

In [ ]:
# @title 2. הורדה אוטומטית של קובצי העברית הנקיים + 290 דגימות הרגש המקוריות באנגלית
import zipfile
import urllib.request

%cd /content
GITHUB_ZIP_URL = 'https://raw.githubusercontent.com/barelgrabli/avatar-hebrew-dubbing/main/dibob_colab_inputs.zip'

print('⬇️ מוריד את dibob_colab_inputs.zip מ-GitHub...')
urllib.request.urlretrieve(GITHUB_ZIP_URL, 'dibob_colab_inputs.zip')
with zipfile.ZipFile('dibob_colab_inputs.zip', 'r') as zf:
    zf.extractall('.')

with open('subtitles/Avatar_Seven_Havens_S01E01_dubbing_script.json', 'r', encoding='utf-8') as f:
    dubbing_script = json.load(f)

with open('voice_refs/speakers_manifest.json', 'r', encoding='utf-8') as f:
    speakers_manifest = json.load(f)

print(f'✅ נטענו {len(dubbing_script)} שורות עברית נקיות + {len(dubbing_script)} דגימות רגש מקוריות באנגלית!')

In [ ]:
# @title 3. מנוע העברת קול ורגש פר-משפט + השוואת שמיעה (לפני / מקור באנגלית / אחרי)
from IPython.display import Audio, display

@torch.no_grad()
def convert_line_with_emotion(dub_id, spk, diffusion_steps=None, cfg_rate=0.75):
    if diffusion_steps is None:
        diffusion_steps = 25 if device.type == 'cuda' else 10
    src_path = f'/content/dubbed_lines_clean_he/dub_{dub_id:03d}.wav'
    orig_line_path = f'/content/orig_lines_en/orig_{dub_id:03d}.wav'
    char_ref_path = f"/content/{speakers_manifest[spk]['ref_audio_path']}"

    source_audio = librosa.load(src_path, sr=sr)[0]
    line_ref = librosa.load(orig_line_path, sr=sr)[0]
    char_ref = librosa.load(char_ref_path, sr=sr)[0]

    # בניית רפרנס משולב: קודם המשפט המקורי מאותה סצנה (לטון ולרגש המדויק) + עוגן קול של הדמות (עד 7 שניות סה"כ)
    if len(line_ref) >= int(1.2 * sr):
        combined_ref = np.concatenate([line_ref[:int(4.0 * sr)], char_ref[:int(3.5 * sr)]])
    else:
        combined_ref = char_ref[:int(6.0 * sr)]

    src_tensor = torch.tensor(source_audio).unsqueeze(0).float().to(device)
    ref_tensor = torch.tensor(combined_ref).unsqueeze(0).float().to(device)
    line_tensor_16k = torchaudio.functional.resample(torch.tensor(line_ref).unsqueeze(0).float().to(device), sr, 16000)

    converted_waves_16k = torchaudio.functional.resample(src_tensor, sr, 16000)
    ori_waves_16k = torchaudio.functional.resample(ref_tensor, sr, 16000)

    S_alt = semantic_fn(converted_waves_16k)
    S_ori = semantic_fn(ori_waves_16k)

    mel = mel_fn(src_tensor)
    mel2 = mel_fn(ref_tensor)
    target_lengths = torch.LongTensor([mel.size(2)]).to(device)
    target2_lengths = torch.LongTensor([mel2.size(2)]).to(device)

    feat2 = torchaudio.compliance.kaldi.fbank(ori_waves_16k, num_mel_bins=80, dither=0, sample_frequency=16000)
    feat2 = feat2 - feat2.mean(dim=0, keepdim=True)
    style2 = campplus_model(feat2.unsqueeze(0))

    # חילוץ עקומת האינטונציה והרגש (F0) מהמשפט המקורי באנגלית ומהמשפט העברי
    F0_ori = torch.from_numpy(f0_fn(ori_waves_16k[0], thred=0.03)).to(device)[None]
    F0_alt = torch.from_numpy(f0_fn(converted_waves_16k[0], thred=0.03)).to(device)[None]
    F0_line = torch.from_numpy(f0_fn(line_tensor_16k[0], thred=0.03)).to(device)[None]

    voiced_ori = F0_ori[F0_ori > 1]
    voiced_alt = F0_alt[F0_alt > 1]
    voiced_line = F0_line[F0_line > 1]

    log_f0_alt = torch.log(F0_alt + 1e-5)
    shifted_log_f0_alt = log_f0_alt.clone()

    if len(voiced_alt) > 2 and len(voiced_ori) > 2:
        med_alt = torch.median(torch.log(voiced_alt + 1e-5))
        std_alt = torch.std(torch.log(voiced_alt + 1e-5)) + 1e-4
        # אם המשפט המקורי באנגלית מכיל קול ברור, ניקח את גובה הקול והמנעד הרגשי שלו!
        if len(voiced_line) > 5:
            med_tgt = 0.65 * torch.median(torch.log(voiced_line + 1e-5)) + 0.35 * torch.median(torch.log(voiced_ori + 1e-5))
            std_tgt = torch.std(torch.log(voiced_line + 1e-5)) + 1e-4
            expressiveness_scale = torch.clamp(std_tgt / std_alt, 1.0, 1.55)
        else:
            med_tgt = torch.median(torch.log(voiced_ori + 1e-5))
            expressiveness_scale = 1.15

        mask = F0_alt > 1
        shifted_log_f0_alt[mask] = med_tgt + (log_f0_alt[mask] - med_alt) * expressiveness_scale

        # שילוב 30% מעקומת המלודיה של המשפט המקורי באנגלית (עלייה בשאלה/צעקה/התרגשות)
        if len(voiced_line) > 8:
            line_contour = torch.log(voiced_line + 1e-5) - torch.median(torch.log(voiced_line + 1e-5))
            line_contour_interp = torch.nn.functional.interpolate(
                line_contour[None, None, :], size=F0_alt.shape[-1], mode='linear', align_corners=True
            ).squeeze()
            shifted_log_f0_alt[mask] += 0.30 * line_contour_interp[mask]

    shifted_f0_alt = torch.exp(shifted_log_f0_alt)
    shifted_f0_alt[F0_alt <= 1] = 0.0

    cond, _, _, _, _ = vc_model.length_regulator(S_alt, ylens=target_lengths, n_quantizers=3, f0=shifted_f0_alt)
    prompt_condition, _, _, _, _ = vc_model.length_regulator(S_ori, ylens=target2_lengths, n_quantizers=3, f0=F0_ori)

    cat_condition = torch.cat([prompt_condition, cond], dim=1)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
        vc_target = vc_model.cfm.inference(
            cat_condition,
            torch.LongTensor([cat_condition.size(1)]).to(device),
            mel2, style2, None, diffusion_steps,
            inference_cfg_rate=cfg_rate
        )
        vc_target = vc_target[:, :, mel2.size(-1):]

    vc_wave = vocoder_fn(vc_target.float()).squeeze().cpu().numpy()
    # Resample to 24kHz mono for our mixer
    vc_wave_24k = librosa.resample(vc_wave, orig_sr=sr, target_sr=24000)
    peak = np.max(np.abs(vc_wave_24k))
    if peak > 1e-4:
        vc_wave_24k = vc_wave_24k * (0.78 / peak)
    return vc_wave_24k

# בדיקת השוואה על 4 משפטים עם רגשות שונים (פאבי מתלהבת, מליקה צועקת/מתחננת, ג'יי מופתע, קארתיק סמכותי)
sample_ids = [8, 77, 242, 62]
for item in dubbing_script:
    if item['dub_id'] in sample_ids:
        did = item['dub_id']
        spk = item['speaker']
        print(f"\n======================================================")
        print(f"🎭 דמות: {spk} | אנגלית: {item['text_en']} | עברית: {item['text_he']}")
        orig_en, _ = sf.read(f'/content/orig_lines_en/orig_{did:03d}.wav')
        clean_he, _ = sf.read(f'/content/dubbed_lines_clean_he/dub_{did:03d}.wav')
        vc_he = convert_line_with_emotion(did, spk)
        print('1️⃣ המשפט המקורי באנגלית (הקול והרגש במקור):')
        display(Audio(orig_en, rate=24000))
        print('2️⃣ לפני Seed-VC (קריינות העברית הרגילה):')
        display(Audio(clean_he, rate=24000))
        print('3️⃣ אחרי Seed-VC (הקול המקורי של הדמות + הרגש בעברית!):')
        display(Audio(vc_he, rate=24000))

In [ ]:
# @title 4. הרצת Seed-VC על כל 290 המשפטים בפרק והורדת dubbed_lines.zip
from tqdm.auto import tqdm
from google.colab import files

os.makedirs('/content/dubbed_lines', exist_ok=True)

for item in tqdm(dubbing_script, desc='Converting 290 lines with Seed-VC + Emotion'):
    did = item['dub_id']
    out_path = f'/content/dubbed_lines/dub_{did:03d}.wav'
    if os.path.exists(out_path):
        continue
    spk = item['speaker']
    try:
        vc_wav = convert_line_with_emotion(did, spk, cfg_rate=0.75)
        sf.write(out_path, vc_wav, 24000)
    except Exception as e:
        print(f'Warning on line {did}, falling back to clean Hebrew: {e}')
        clean_he, _ = sf.read(f'/content/dubbed_lines_clean_he/dub_{did:03d}.wav')
        sf.write(out_path, clean_he, 24000)

%cd /content
!zip -r -q dubbed_lines.zip dubbed_lines/
print('✅ מוכן! מוריד את dubbed_lines.zip למחשב שלך...')
files.download('dubbed_lines.zip')